In [ ]:
import requests
import json
import time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from datetime import timedelta
from datetime import datetime

import os

out_directory = 'Data'
if not os.path.exists(out_directory):
    os.makedirs(out_directory)

In [ ]:
from credentials import *

In [ ]:
hash_tags = ['Booksounds','Pageflip','Pageflipping','Bookspinebreaking','Bookspine','Bookasmr','Paperasmr','Tabbing','Annotating','Dogearing','Prettybooks','Sprayededges','Bookhaul','Bookunboxing','Shelfie','Bookshelfie','Bookshelf','Bookstack','Bookunboxing','Foreedgepainting']
fields = "id,video_description,create_time,region_code,share_count,view_count,like_count,comment_count,music_id,hashtag_names,username,effect_ids,playlist_id,voice_to_text,is_stem_verified,favorites_count,video_duration,hashtag_info_list,sticker_info_list,effect_info_list,video_mention_list,video_label,video_tag"

start_date = "20220101"
end_date = "20250101"

In [ ]:
base_url = 'https://open.tiktokapis.com/v2/oauth/token/'

headers = {'Content-Type': 'application/x-www-form-urlencoded', 
        'Cache-Control': 'no-cache'}

data = {"client_key":client_key,
        "client_secret":client_secret,
        "grant_type":"client_credentials"}

response = requests.post(base_url, headers=headers, data=data)

response_as_json = response.json()
access_token = response_as_json["access_token"]
print(access_token)

In [ ]:
url = f"https://open.tiktokapis.com/v2/research/video/query/?fields={fields}"

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

max_count = 100

In [ ]:
def get_videos(hash_tag, max_count, start_date, end_date, headers, search_id, cursor):
    query_params = {
                    "query": {
                        "and": [
                            {"operation": "EQ", 
                             "field_name": "hashtag_name", 
                             "field_values": [hash_tag]}
                        ]
                    },
                    "max_count": max_count,
                    "start_date": start_date,
                    "end_date": end_date,
                    "cursor": cursor,
                    "search_id":search_id
                    }
    

    response = requests.post(url, headers=headers, data=json.dumps(query_params))
    print(f"HTTP Status code: {response.status_code}")
    json_data = response.json()
    return json_data



def generate_date_range(start_date, end_date): 

    
    start = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")
    delta = end - start
    current_date = start
    
    dates = []
    temp_end_date = current_date
    while current_date <= end:
        delta_end = end-current_date
        if delta_end.days <= 30:
            dates.append((current_date.strftime("%Y-%m-%d").replace("-",""), end.strftime("%Y-%m-%d").replace("-","")))
            break
        else:
            temp_end_date += timedelta(days=29)
            dates.append((current_date.strftime("%Y-%m-%d").replace("-",""),
                        temp_end_date.strftime("%Y-%m-%d").replace("-","")))
            current_date = temp_end_date + timedelta(days=1)
        
    return dates

In [ ]:
data = []
all_columns = []
date_ranges = generate_date_range(start_date, end_date)

for hash_tag in hash_tags:
    
    path = os.path.join(out_directory , f"{hash_tag.lower().strip()}.tsv")
    if not os.path.exists(path):
    
        out = open(path,'w',encoding='utf-8')

        print(f'\n{hash_tag}')

        for date_range in date_ranges: 

            search_id=""
            cursor = 0
            has_more = True

            start = date_range[0]
            end = date_range[1]

            print(f"Date range: {start}-{end}")

            while has_more == True:

                try:
                    columns = []
                    hashtag_data = get_videos(hash_tag, max_count, start, end, headers, search_id, cursor)

                    print("error code and message:",hashtag_data["error"]["code"], hashtag_data["error"]["message"])
                    print("cursor:",hashtag_data["data"]["cursor"])
                    print("has_more:",hashtag_data["data"]["has_more"])
                    print("_"*10)
                    
                    cursor = hashtag_data["data"]["cursor"]
                    has_more = hashtag_data["data"]["has_more"]
                    search_id = hashtag_data["data"]["search_id"]

                    df = pd.DataFrame(hashtag_data["data"]["videos"])
                    columns = df.columns.tolist()
                    all_columns.extend(columns)
                    all_columns = list(set(all_columns))
                    data.append(df)
                    time.sleep(10)
                except:
                    print('An error occurred ... ')

        out = open(os.path.join( out_directory , f"{hash_tag.lower().strip()}.tsv" ),'w',encoding='utf-8')

        out.write("\t".join(all_columns))

        for df in data:
            for i,row in df.iterrows():
                for column_nr,column in enumerate(all_columns):

                    if column_nr == 0:
                        out.write('\n')
                    else:
                        out.write('\t')

                    if column in row:

                        out.write(f'"{row[column]}"')

        out.close()
